# 06 — Recursos Avançados do Delta Lake + Observabilidade
## Módulo 4 | Seções do Exame: S1 (6%) + S6 (10%)

**O que este notebook faz:**
- **Time Travel** — consultas históricas e rollback por versão
- **Change Data Feed (CDF)** — capturar INSERTs, UPDATEs e DELETEs
- **OPTIMIZE + Z-ORDER** — compactação e co-localização de dados
- **VACUUM** — limpeza física de arquivos antigos
- **Spark UI** — identificar data skew e gargalos de performance
- **AQE** — Adaptive Query Execution
- **Liquid Clustering** — alternativa moderna ao Z-ORDER

Cada seção é **independente** — você pode executá-las separadamente para revisão.

---

### ✅ Checklist deste notebook

**Time Travel:**
- [ ] `DESCRIBE HISTORY` listou versões com timestamps
- [ ] `VERSION AS OF 0` retornou dados da versão inicial
- [ ] `RESTORE TABLE` executado — rollback criou nova versão
- [ ] Entendi: consulta histórica ≠ RESTORE (RESTORE cria nova versão)

**Change Data Feed:**
- [ ] CDF habilitado na tabela Silver
- [ ] `table_changes()` listou operações com `_change_type`
- [ ] Entendi: insert / update_preimage / update_postimage / delete

**OPTIMIZE + Z-ORDER:**
- [ ] `DESCRIBE DETAIL` antes: anotei número de arquivos
- [ ] `OPTIMIZE ZORDER BY` executado
- [ ] `DESCRIBE DETAIL` depois: número de arquivos reduziu
- [ ] Entendi: Z-ORDER = co-localização, não ordenação física

**VACUUM:**
- [ ] Entendi: VACUUM apaga fisicamente arquivos de versões antigas
- [ ] `VACUUM RETAIN 168 HOURS` executado
- [ ] Entendi a relação VACUUM ↔ janela de Time Travel

**Spark UI + AQE + Liquid Clustering:**
- [ ] Spark UI aberto — identifiquei colunas `Shuffle Read Size`
- [ ] AQE habilitado e testado
- [ ] Liquid Clustering aplicado — entendi diferença vs Z-ORDER

In [ ]:
dbutils.widgets.text("catalog", "workshop_weather", "Catálogo UC")
CATALOG      = dbutils.widgets.get("catalog")
SILVER_CW    = f"{CATALOG}.silver.current_weather"
SILVER_FC    = f"{CATALOG}.silver.forecast"

print(f"Tabela Silver (current weather) : {SILVER_CW}")
print(f"Tabela Silver (forecast)         : {SILVER_FC}")

---
# SEÇÃO 1 — Time Travel

**Conceito:** O Delta Lake mantém um **transaction log** com todas as versões da tabela.  
Você pode consultar qualquer versão passada sem mover dados — o log aponta para os arquivos corretos.

**Analogia:** É como o histórico de commits do Git — você pode fazer `git checkout` de qualquer versão sem apagar o trabalho atual.

In [ ]:
%sql
-- Lista todas as versões da tabela com operação, timestamp e número de arquivos afetados
DESCRIBE HISTORY workshop_weather.silver.current_weather;

In [ ]:
%sql
-- Consulta a versão 0 (estado após a primeira gravação)
-- NÃO modifica a tabela — apenas lê o estado histórico
SELECT city_name, temp, weather_timestamp, _processed_timestamp
FROM workshop_weather.silver.current_weather VERSION AS OF 0
ORDER BY city_name;

In [ ]:
# Time Travel via PySpark — equivalente ao VERSION AS OF em SQL
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table(SILVER_CW)
)
print(f"Registros na versão 0: {df_v0.count()}")
df_v0.select("city_name", "temp", "weather_timestamp").show()

In [ ]:
%sql
-- RESTORE: reverte a tabela para o estado de uma versão anterior
-- ATENÇÃO: isso MODIFICA a tabela (cria uma nova versão com o estado antigo)
-- Use apenas para demonstração — em produção, RESTORE é um rollback real

-- Primeiro, veja qual é a versão mais recente (substitua N pelo número)
-- RESTORE TABLE workshop_weather.silver.current_weather TO VERSION AS OF 1;

-- Após o RESTORE, DESCRIBE HISTORY mostrará uma nova versão com operation = RESTORE
SELECT 'Execute o RESTORE acima (descomente) para testar o rollback' AS instrucao;

---
# SEÇÃO 2 — Change Data Feed (CDF)

**Conceito:** O CDF captura cada INSERT, UPDATE e DELETE que acontece na tabela Delta,  
expondo-os como um stream de mudanças que pode ser lido por pipelines downstream.

**Casos de uso:**
- Processar apenas os dados **novos ou modificados** desde a última execução (incremental)
- Replicar mudanças para outros sistemas
- Base do Desafio 1 (watermark incremental)

In [ ]:
%sql
-- Confirma que CDF está habilitado (foi ativado no notebook 03 via TBLPROPERTIES)
SHOW TBLPROPERTIES workshop_weather.silver.current_weather;

In [ ]:
%sql
-- Habilita CDF se ainda não estiver ativo
ALTER TABLE workshop_weather.silver.current_weather
SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true');

In [ ]:
%sql
-- Lê as mudanças a partir da versão 1
-- _change_type: insert | update_preimage | update_postimage | delete
SELECT
    _change_type,
    _commit_version,
    _commit_timestamp,
    city_name,
    temp,
    weather_timestamp
FROM table_changes('workshop_weather.silver.current_weather', 1)
ORDER BY _commit_version, _change_type;

In [ ]:
# CDF via PySpark — útil para pipelines de streaming incremental
df_changes = (
    spark.read
    .format("delta")
    .option("readChangeData", "true")
    .option("startingVersion", 1)
    .table(SILVER_CW)
)

print("Tipos de mudanças capturadas pelo CDF:")
df_changes.groupBy("_change_type", "_commit_version").count().orderBy("_commit_version").show()

print()
print("""Legenda dos _change_type:
  insert           → novo registro inserido
  update_preimage  → estado ANTES do UPDATE (valor antigo)
  update_postimage → estado DEPOIS do UPDATE (valor novo)
  delete           → registro removido
""")

---
# SEÇÃO 3 — OPTIMIZE + Z-ORDER

**Problema que o OPTIMIZE resolve:**  
Cada MERGE INTO, append ou UPDATE gera pequenos arquivos Parquet.  
Queries que leem muitos arquivos pequenos são mais lentas do que ler poucos arquivos grandes.

**O que Z-ORDER faz:**  
Co-localiza linhas com os mesmos valores nas colunas especificadas no mesmo arquivo físico.  
Queries com `WHERE city_id = X AND weather_timestamp BETWEEN ...` leem muito menos arquivos.

In [ ]:
%sql
-- Estado ANTES do OPTIMIZE: anote o numFiles
SELECT name, numFiles, ROUND(sizeInBytes / 1024, 2) AS size_kb
FROM (DESCRIBE DETAIL workshop_weather.silver.current_weather);

In [ ]:
%sql
-- OPTIMIZE compacta arquivos pequenos
-- ZORDER BY co-localiza dados pelas colunas de filtro mais frequentes
OPTIMIZE workshop_weather.silver.current_weather
ZORDER BY (city_id, weather_timestamp);

In [ ]:
%sql
-- Estado DEPOIS do OPTIMIZE: compare o numFiles com o anterior
SELECT name, numFiles, ROUND(sizeInBytes / 1024, 2) AS size_kb
FROM (DESCRIBE DETAIL workshop_weather.silver.current_weather);

---
# SEÇÃO 4 — VACUUM

**Conceito:**  
O OPTIMIZE marca arquivos antigos como "deletados" no transaction log, mas não os remove fisicamente.  
O **VACUUM** é quem apaga fisicamente os arquivos que não são mais necessários.

**Relação VACUUM ↔ Time Travel:**  
Se você rodar `VACUUM RETAIN 24 HOURS`, o Time Travel ficará disponível apenas para as últimas 24h.  
O padrão do Delta é 7 dias (`168 HOURS`) — é o balance entre espaço em disco e janela de rollback.

In [ ]:
%sql
-- DRY RUN: mostra o que seria removido sem de fato remover
VACUUM workshop_weather.silver.current_weather
RETAIN 168 HOURS  -- preserva 7 dias de Time Travel
DRY RUN;

In [ ]:
%sql
-- VACUUM de fato — remove arquivos físicos mais antigos que 168h
-- Em produção, execute periodicamente para controlar o tamanho do storage
VACUUM workshop_weather.silver.current_weather
RETAIN 168 HOURS;

---
# SEÇÃO 5 — Spark UI, AQE e Liquid Clustering *(Seção 6 do exame)*

## 5.1 — Lendo a Spark UI

A **Spark UI** mostra o que está acontecendo dentro de cada job Spark executado.  
Para acessar: clique em **View** no output de uma célula que executou Spark, ou acesse via `Compute > <cluster> > Spark UI`.

**O que observar para identificar data skew:**  
Em `Stages > Tasks`, observe as colunas:
- `Shuffle Read Size` — tamanho do shuffle por task
- Se `Max >> Median` (ex: Max = 5GB, Median = 400MB) → **data skew** em uma partição

Execute a query abaixo e depois abra a Spark UI para inspecionar.

In [ ]:
# Query com join — gera shuffle — boa para inspecionar na Spark UI
df_joined = (
    spark.table(SILVER_CW)
    .join(
        spark.table(SILVER_FC).select("city_id", "forecast_timestamp", "temp", "pop"),
        on="city_id",
        how="inner"
    )
    .groupBy("city_name", "country")
    .agg(
        F.count("*").alias("total_forecasts"),
        F.round(F.avg("pop"), 3).alias("avg_pop")
    )
)

from pyspark.sql import functions as F
df_joined.show()

print()
print("👆 Abra a Spark UI (botão 'View' acima) e navegue para:")
print("   Jobs > (último job) > Stages > Tasks")
print("   Observe as colunas: Shuffle Read Size, Duration")
print("   Se Max >> Median em Shuffle Read → indício de data skew")

## 5.2 — AQE (Adaptive Query Execution)

**O que o AQE faz automaticamente:**
1. **Coalesce de partições vazias** — reduz o número de tasks sem dados
2. **Conversão sort-merge → broadcast join** — quando um lado do join é pequeno o suficiente
3. **Skew join handling** — divide automaticamente partições com muito mais dados que a mediana

In [ ]:
# Verifica o estado atual do AQE
current = spark.conf.get("spark.sql.adaptive.enabled")
print(f"AQE atual: {current}")

# Habilita AQE explicitamente
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("✅ AQE habilitado.")
print()
print("Re-execute a célula de join acima e compare na Spark UI:")
print("  - Número de stages deve diminuir (partições vazias coalesced)")
print("  - Sort-merge join pode ser convertido para broadcast join")

In [ ]:
# Re-executa o join com AQE ativo para comparar
df_joined_aqe = (
    spark.table(SILVER_CW)
    .join(
        spark.table(SILVER_FC).select("city_id", "forecast_timestamp", "pop"),
        on="city_id",
        how="inner"
    )
    .groupBy("city_name")
    .agg(F.count("*").alias("n"), F.avg("pop").alias("avg_pop"))
)
df_joined_aqe.show()
print("\nCompare o número de tasks e stages na Spark UI vs a execução anterior (sem AQE).")

## 5.3 — Liquid Clustering

**Liquid Clustering vs Z-ORDER:**

| | Z-ORDER | Liquid Clustering |
|---|---|---|
| Ativação | `OPTIMIZE ... ZORDER BY` | `CLUSTER BY (col)` na criação |
| Manutenção | Requer `OPTIMIZE` periódico manual | Incremental — aplicado automaticamente |
| Flexibilidade | Colunas fixas no OPTIMIZE | Pode alterar colunas com `ALTER TABLE` |
| Produção | Legado — ainda válido | Recomendado para tabelas novas |

**Quando usar:** Tabelas novas com padrões de acesso definidos por 1-4 colunas.

In [ ]:
%sql
-- Aplica Liquid Clustering na tabela Silver forecast
-- Isso define as colunas de clustering — o OPTIMIZE vai aplicar incrementalmente
ALTER TABLE workshop_weather.silver.forecast
CLUSTER BY (city_id, forecast_timestamp);

In [ ]:
%sql
-- Confirma que o Liquid Clustering foi aplicado
DESCRIBE DETAIL workshop_weather.silver.forecast;

In [ ]:
%sql
-- OPTIMIZE sem ZORDER — aplica o Liquid Clustering definido acima
OPTIMIZE workshop_weather.silver.forecast;

---
## ✅ Checklist Final — Módulo 4 concluído?

**Time Travel:**
- [ ] `DESCRIBE HISTORY` mostrou versões com timestamps e operações
- [ ] Consultei `VERSION AS OF 0` — dados históricos retornados corretamente
- [ ] Entendi: consulta histórica não modifica a tabela; RESTORE cria nova versão

**CDF:**
- [ ] `table_changes()` retornou operações com `_change_type`
- [ ] Sei ler `update_preimage` (antes) vs `update_postimage` (depois)

**OPTIMIZE + Z-ORDER:**
- [ ] `numFiles` reduziu após OPTIMIZE
- [ ] Entendi: Z-ORDER = co-localização, não ordenação global

**VACUUM:**
- [ ] DRY RUN mostrou arquivos que seriam removidos
- [ ] VACUUM executado com RETAIN 168 HOURS
- [ ] Entendi: VACUUM apaga fisicamente → limita a janela de Time Travel

**Spark UI + AQE + Liquid Clustering:**
- [ ] Abri a Spark UI e inspecionei o stage com join
- [ ] AQE habilitado — comparei execução com/sem
- [ ] Liquid Clustering aplicado na tabela forecast

**Próximo: `06b_governance_uc.ipynb`**

---
*Workshop: Engenharia de Dados com Databricks & Delta Lake — OpenWeatherMap API*